In [ ]:
def escritura(movimiento:list,angulo,vel,ff=0.35,np=1):
    """La funcion escritura sirve para transformar archivos gz en csv

    Args:
        itera (int): Es la posición de los datos con indices 0,1,2,...,N siendo N la cantiad de archivos en la carpeta contenedora
        paso (int): Es el espaciado con el que se guaran los datos
        movimiento (list): Es el movimiento x,y,z del proyectil
        angulo (float): Es el ángulo con el que impacta el proyectil
        vel (float): velocidad en z del proyectil
        ff (float, optional): Es el  factor de llenado de la cama con la que se simula. Defaults to 0.35.
        np (int, optional): La cantidad de partículas usadas en el proyectil. Defaults to 1.
    """
    from gzip import open as opengz
    from os import mkdir, listdir,path
    ## Definimos la carpeta donde estan los outputs de la sim.
    ruta = f"./erosion_bed_SI_ff_{ff}_np_{np}_vel_{vel}_angle_{angulo}_move_{movimiento[0]}_{movimiento[1]}_{movimiento[2]}"
    for archivo in listdir(ruta):
        if archivo.startswith("output.") and ".gz" in archivo:
            nombre = archivo.replace(".gz","")
            if  not path.isfile(f"{ruta}/{nombre}/{nombre}.csv"): ## Ver si no lo vonvertimos antes
                mkdir(f"{ruta}/{nombre}")
                ## Escirbir archivo ....
                with opengz(f'{ruta}/{archivo}', 'rt', newline='') as csv_file:
                    csv_data = csv_file.read()
                    with open(f'{ruta}/{nombre}/{nombre}.csv', 'wt') as out_file:
                        out_file.write(csv_data)
                print(f"Escritura de: {ruta}/{nombre}/{nombre}.csv\nFue exitosa :)")
            else:
                print(f"Ya existe el archivo {ruta}/{nombre}/{nombre}.csv")
def lectura(itera,paso,movimiento:list,angulo,vel,ff=0.35,np=1):
    """La funcion lectura sirve para abir archivos csv

    Args:
        itera (int): Es la posición de los datos con indices 0,1,2,...,N siendo N la cantiad de archivos en la carpeta contenedora
        paso (int): Es el espaciado con el que se guaran los datos
        movimiento (list): Es el movimiento x,y,z del proyectil
        angulo (float): Es el ángulo con el que impacta el proyectil
        vel (float): velocidad en z del proyectil
        ff (float, optional): Es el  factor de llenado de la cama con la que se simula. Defaults to 0.35.
        np (int, optional): La cantidad de partículas usadas en el proyectil. Defaults to 1.

    Returns:
        pandas.DataFrame: Devuelve un data frame con los datos ["id", "type", "x", "y", "z", "vx", "vy", "vz", "omegax", "omegay", "omegaz", "c_1"]
    """
    from pandas import read_csv
    return read_csv(f"./erosion_bed_SI_ff_{ff}_np_{np}_vel_{vel}_angle_{angulo}_move_{movimiento[0]}_{movimiento[1]}_{movimiento[2]}/output.{paso*itera}/output.{paso*itera}.csv",skiprows=9,names=["id", "type", "x", "y", "z", "vx", "vy", "vz", "omegax", "omegay", "omegaz", "c_1"],delimiter=" ")
def check(movimiento:list,angulo,vel,ff=0.35,np=1)->int:
    """Esta funcion cuenta la cantidad de archivos que se transformaron en csv
    Args:
        movimiento (list): Es el movimiento x,y,z del proyectil
        angulo (float): Es el ángulo con el que impacta el proyectil
        vel (float): velocidad en z del proyectil
        ff (float, optional): Es el  factor de llenado de la cama con la que se simula. Defaults to 0.35.
        np (int, optional): La cantidad de partículas usadas en el proyectil. Defaults to 1.
    Returns:
        int: Cantidad de carpetas que hay en el directorio de la simulación
        Paso (int): Cantidad de pasos que salta entre archivos
    """
    from os import listdir, path 
    ruta = f"./erosion_bed_SI_ff_{ff}_np_{np}_vel_{vel}_angle_{angulo}_move_{movimiento[0]}_{movimiento[1]}_{movimiento[2]}"
    carpetas = [d for d in listdir(ruta) if d.startswith("output.") and not ".gz" in d]
    if not carpetas:
        print("No hay datos registrados todavía.")
        return 0
    return len(carpetas)

In [ ]:
import gc
from numpy import sum as Sumatoria
from numpy import mean , std

movimiento = [0,0,0]
angulo = 0 
vel = -10
threshold = 0.004

## Obtengo la cantidad de archivos y los saltos de las simulaciones
# Nota: checkparquet y lecturaparquet son las funciones que definiste previamente
LenTotal, paso = check(movimiento,angulo,vel), 50000

## Tiro toda la grabage a René Descart
gc.collect(2)

## Inicializo mi análisis 
Data_collect = {"K":[],"Dispersión":[],"Max":[],"Min":[],"Vcm":[],"N":0,"Eyección":[]}

for i in range(LenTotal):
    ## Extraigo mis datos
    Datos = lectura(i,paso,movimiento,angulo,vel)
    
    ## Busco la altura de la cama
    if i==0:
        altura = Datos[Datos["type"]==1]["z"].max() ## Esta variable NO HAY QUE BORRARLA
    
    ## En caso de que no este el proyectil en output.0 lo pongo a mano jeje
    if Datos[Datos["type"]==2].empty and i==0:
        v0 = vel
        N = len(Datos["type"])+1
        Data_collect["N"] = N
    ## Esto es para añadir el proyectil unicamente 
    else:
        v0=0    
    
    ## Calculo la velocidad del CM
    Data_collect["Vcm"].append([Sumatoria(Datos["vx"])/N, Sumatoria(Datos["vy"])/N, (Sumatoria(Datos["vz"])+v0)/N])
    
    ## Calculo el módulo de velocidad relativa a la velocidad del CM 
    Vnorm2 = (Datos["vx"]-Data_collect["Vcm"][i][0])**2 + (Datos["vy"]-Data_collect["Vcm"][i][1])**2 + (Datos["vz"]-Data_collect["Vcm"][i][2])**2
    
    ## Calculo la energía cinética de la cama (para añadir la de la partícula, a t=0 sumar 0.5*v_0^2)
    Data_collect["K"].append(Sumatoria(Vnorm2)*0.5)
    
    ## Busco las velocidades máximas y mínimas
    Data_collect["Max"].append(max(Vnorm2))
    Data_collect["Min"].append(min(Vnorm2))
    
    ## Pongo un filtro para buscar las partículas eyectadas
    filtro = (Datos["z"]>altura) & (Datos["vz"]>threshold)
    Data_collect["Eyección"].append(filtro.sum())
    
    ## Limpio la garbage
    del Datos, Vnorm2, filtro
    gc.collect()